# Geometry Working Memory

https://www.biorxiv.org/content/10.64898/2026.08.31.748237v1

In [ ]:
# libraries

import os
import shutil
import numpy as np
import pickle
import hypertools as hyp
import warnings


In [ ]:
# functions

def procrustes_with_transformation(source, target, scaling=True, reflection=True, reduction=False,
               oblique=False, oblique_rcond=-1, format_data=True, return_transform=True):
    """
    Procrustes transformation that can optionally return the transformation matrix.

    Parameters
    ----------
    return_transform : bool
        If True, return the transformation matrix in addition to aligned data.

    Returns
    ----------
    aligned_source : ndarray
        Source aligned to target's space.

    proj : ndarray (only if return_transform=True)
        Transformation matrix to apply to other data.
    """

    # Now import the necessary modules
    from hypertools._externals.srm import SRM
    from hypertools.tools.procrustes import procrustes
    from hypertools.tools.format_data import format_data as formatter
    from hypertools._shared.helpers import memoize

    def fit(source, target):
        datas = (source, target)
        sn, sm = source.shape
        tn, tm = target.shape

        if sn != tn:
            raise ValueError("Source and target must have same number of samples")

        # Sums of squares
        ssqs = [np.sum(d**2, axis=0) for d in datas]
        for i in range(2):
            if np.all(ssqs[i] <= np.abs((np.finfo(datas[i].dtype).eps * sn )**2)):
                raise ValueError("Invariance in time not handled")

        norms = [np.sqrt(np.sum(ssq)) for ssq in ssqs]
        normed = [data / norm for (data, norm) in zip(datas, norms)]

        if sm < tm:
            normed[0] = np.hstack((normed[0], np.zeros((sn, tm - sm))))
        if sm > tm:
            if reduction:
                normed[1] = np.hstack((normed[1], np.zeros((sn, sm - tm))))
            else:
                raise ValueError("reduction=False and target has fewer features")

        source_n, target_n = normed
        if oblique:
            if sn == sm and tm == 1:
                T = np.linalg.solve(source_n, target_n)
            else:
                T = np.linalg.lstsq(source_n, target_n, rcond=oblique_rcond)[0]
            ss = 1.0
        else:
            U, s, Vh = np.linalg.svd(np.dot(target_n.T, source_n), full_matrices=False)
            T = np.dot(Vh.T, U.T)

            if not reflection:
                nsv = len(s)
                s[:-1] = 1
                s[-1] = np.linalg.det(T)
                T = np.dot(U[:, :nsv] * s, Vh)

            ss = sum(s)

        # if sm != tm:
        #     T = T[:sm, :tm]

        if scaling:
            scale = ss * norms[1] / norms[0]
            proj = scale * T
        else:
            proj = T

        return proj

    def transform(data, proj):
        return (np.asmatrix(data) * proj).A

    if format_data:
        source, target = formatter([source, target])

    proj = fit(source, target)
    aligned = transform(source, proj)

    if return_transform:
        return aligned, proj
    else:
        return aligned


def align_noscaling_with_transformation(data, align='hyper', normalize=None, ndims=None, method=None,
          format_data=True):
    """
    Same funciton that hypertools.align, but also returning transformation matrices.

    Aligns a list of arrays and returns both aligned data and transformation matrices.

    This function takes a list of high-dimensional arrays and 'hyperaligns' them
    to a 'common' space, or coordinate system following the approach outlined by
    Haxby et al, 2011. Hyperalignment uses linear transformations (rotation,
    reflection, translation, scaling) to register a group of arrays to a common
    space. This can be useful when two or more datasets describe an identical
    or similar system, but may not be in same coordinate system.

    Parameters
    ----------
    data : numpy array, pandas df, or list of arrays/dfs
        A list of Numpy arrays or Pandas Dataframes

    align : str or dict
        If str, either 'hyper' or 'SRM'. If 'hyper', alignment algorithm will be
        hyperalignment. If 'SRM', alignment algorithm will be shared response
        model.

    format_data : bool
        Whether or not to first call the format_data function (default: True).

    Returns
    ----------
    aligned : list
        An aligned list of numpy arrays
    transforms : list
        A list of transformation matrices used for alignment
    """

    # Now import the necessary modules
    from hypertools._externals.srm import SRM
    from hypertools.tools.procrustes import procrustes
    from hypertools.tools.format_data import format_data as formatter
    from hypertools._shared.helpers import memoize

    # if model is None, just return data
    if align is None:
        return data, []

    elif isinstance(align, dict):
        if align['model'] is None:
            return data, []

    else:
        if method is not None:
            warnings.warn('The method argument will be deprecated.  Please use align.')
            align = method

        if align is True:
            warnings.warn("Setting align=True will be deprecated. Please specify the \
                          type of alignment, i.e. align='hyper'.")
            align = 'hyper'

        # common format
        if format_data:
            data = formatter(data, ppca=True)

        if len(data) == 1:
            warnings.warn('Data in list of length 1 cannot be aligned. Skipping the alignment.')

        if data[0].shape[1] >= data[0].shape[0]:
            warnings.warn('The number of features exceeds number of samples. This can lead \
                         to overfitting. We recommend reducing the dimensionality.')

        if (align == 'hyper') or (method == 'hyper'):

            ##STEP 0: STANDARDIZE SIZE AND SHAPE##
            sizes_0 = [x.shape[0] for x in data]
            sizes_1 = [x.shape[1] for x in data]

            # Find the smallest number of rows
            R = min(sizes_0)
            C = max(sizes_1)

            m = [np.empty((R, C), dtype=np.ndarray)] * len(data)

            for idx, x in enumerate(data):
                y = x[0:R, :]
                missing = C - y.shape[1]
                add = np.zeros((y.shape[0], missing))
                y = np.append(y, add, axis=1)
                m[idx] = y

            ##STEP 1: TEMPLATE##
            for x in range(0, len(m)):
                if x == 0:
                    template = np.copy(m[x])
                else:
                    next, transf = procrustes_with_transformation(m[x], template / (x + 1), scaling=False)
                    template += next
            template /= len(m)

            ##STEP 2: NEW COMMON TEMPLATE##
            # Align each subject to the template from STEP 1
            template2 = np.zeros(template.shape)
            for x in range(0, len(m)):
                next, transf = procrustes_with_transformation(m[x], template, scaling=False)
                template2 += next
            template2 /= len(m)

            # STEP 3: ALIGN TO NEW TEMPLATE
            aligned = [np.zeros(template2.shape)] * len(m)
            transforms = []  # List to store transformation matrices
            for x in range(0, len(m)):
                next, transf = procrustes_with_transformation(m[x], template2, scaling=False)
                aligned[x] = next
                transforms.append(transf)  # Store each transformation matrix
            return aligned, transforms

        elif (align == 'SRM') or (method == 'SRM'):
            
            data = [i.T for i in data]
            srm = SRM(features=np.min([i.shape[0] for i in data]))
            fit = srm.fit(data)
            transformed_data = [i.T for i in srm.transform(data)]
            return transformed_data, fit.components_  # Return transformation matrices from SRM




In [ ]:
### set paths and settings

path_root = '/path_to_local'

# settings

subjects = [f'sub_{i:02d}' for i in range(1, 50)]

time_windows = ['encode', 'maint', 's2']
folder_input = 'X_matrix'
pca_folder = 'pca_noaligned'
pca_aligned_folder = 'pca_aligned'
number_components_align = 10

outputs_correct_trials = [
'control_2gratings_2polygons',                                  'control_1gratings_1polygons',
'update_2gratings_relevant_2polygons_nonrelevant',              'update_1gratings_relevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_relevant',              'update_1gratings_nonrelevant_1polygons_relevant',
'inhibition_2gratings_relevant_2polygons_nonrelevant',          'inhibition_1gratings_relevant_1polygons_nonrelevant',
'inhibition_2gratings_nonrelevant_2polygons_relevant',        'inhibition_1gratings_nonrelevant_1polygons_relevant',
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      'update_1gratings_nolongerrelevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',      'update_1gratings_nonrelevant_1polygons_nolongerrelevant',
]

outputs_incorrect_trials = [
'control_2gratings_2polygons',                                  
'update_2gratings_relevant_2polygons_nonrelevant',              
'update_2gratings_nonrelevant_2polygons_relevant',              
'inhibition_2gratings_relevant_2polygons_nonrelevant',          
'inhibition_2gratings_nonrelevant_2polygons_relevant',        
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',       
]



# Hyperalignment - correct trials

In [ ]:
### hyperalignment

trial_type = 'correct_trials'

if trial_type == 'correct_trials':
    outputs = outputs_correct_trials
elif trial_type == 'incorrect_trials':
    outputs = outputs_incorrect_trials

for time_window in time_windows:

    if time_window == 'encode':

        segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

    elif time_window == 'maint':

        segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

    elif time_window == 's2':

        segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

    # temporary storage folder
    path_tmp = os.path.join(path_root, 'results', pca_aligned_folder, 'tmp_folder_scores', time_window + '_time_resolved')
    if not os.path.isdir(path_tmp):
        os.makedirs(path_tmp)

    for output in outputs:

        print('hyperalignment ' + time_window + ' ' + output, flush=True)

        for time_segment in segments:

            key = time_window + '_segment' + str(time_segment) + '_' + output
                    
            pc_scores_list = []

            # append PC scores across subjects
            for sub_i in subjects:

                # load PC scores matrix
                path_pca_sub =   os.path.join(path_root, 'results', pca_folder, time_window + '_time_resolved', sub_i)
                filename = os.path.join(path_pca_sub, 'pca_' + trial_type + '.pkl')
                with open(filename, 'rb') as file:
                    pca_dict = pickle.load(file)
                pc_scores_list.append(pca_dict['pc_scores_' + key][:,0:number_components_align]) # append conditions by channels (PC scores)

            # hyperalignment 
            aligned, transforms = align_noscaling_with_transformation(pc_scores_list, align='hyper', normalize=None, ndims=None, method=None, format_data=False)

            # temporary storage
            if not os.path.isdir(os.path.join(path_tmp, key)):
                os.makedirs(os.path.join(path_tmp, key))
            filename = os.path.join(path_tmp, key, 'aligned.pkl')
            with open(filename, "wb") as file:
                pickle.dump(aligned, file)
            filename = os.path.join(path_tmp, key, 'transforms.pkl')
            with open(filename, "wb") as file:
                pickle.dump(transforms, file)


### reformat outputs

for time_window in time_windows:

    if time_window == 'encode':

        segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

    elif time_window == 'maint':

        segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

    elif time_window == 's2':

        segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

    for sub_idx, sub_i in enumerate(subjects):

        pc_scores_dict = {}
        transforms_dict = {}

        path_tmp = os.path.join(path_root, 'results', pca_aligned_folder, 'tmp_folder_scores', time_window + '_time_resolved')
        path_output_sub = os.path.join(path_root, 'results', pca_aligned_folder, time_window + '_time_resolved', sub_i)
        os.makedirs(path_output_sub, exist_ok=True)

        # loop over outputs
        for output in outputs:

            print('reformat ' + time_window + ' ' + output, flush=True)

            for time_segment in segments:
                    
                key = time_window + '_segment' + str(time_segment) + '_' + output

                filename = os.path.join(path_tmp, key, 'aligned.pkl')
                with open(filename, 'rb') as file:
                    aligned = pickle.load(file)
                filename = os.path.join(path_tmp, key, 'transforms.pkl')
                with open(filename, 'rb') as file:
                    transforms = pickle.load(file)

                path_pca_sub =   os.path.join(path_root, 'results', pca_folder, time_window + '_time_resolved', sub_i)
                filename = os.path.join(path_pca_sub, 'pca_' + trial_type + '.pkl')
                with open(filename, 'rb') as file:
                    pca_dict = pickle.load(file)
                pc_scores_dict['eigvecs_' + key] = pca_dict['eigvecs_' + key][:,0:number_components_align]

                pc_scores_dict['pc_scores_' + key] = aligned[sub_idx]
                transforms_dict[key] = transforms[sub_idx]

        # Save the dictionary to a file using pickle
        with open(os.path.join(path_output_sub, 'pca_' + trial_type + '.pkl'), 'wb') as file:
            pickle.dump(pc_scores_dict, file)

        with open(os.path.join(path_output_sub, 'transforms_' + trial_type + '.pkl'), 'wb') as file:
            pickle.dump(transforms_dict, file)

## delete tmp_folder_scores

path_tmp = os.path.join(path_root, 'results', pca_aligned_folder, 'tmp_folder_scores')

shutil.rmtree(path_tmp)


# Hyperalignment - incorrect trials

Apply transforms from correct trials to incorrect trials

In [16]:
### hyperalignment

trial_type = 'incorrect_trials'

if trial_type == 'correct_trials':
    outputs = outputs_correct_trials
elif trial_type == 'incorrect_trials':
    outputs = outputs_incorrect_trials

for time_window in time_windows:

    if time_window == 'encode':

        segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

    elif time_window == 'maint':

        segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

    elif time_window == 's2':

        segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

    for sub_idx, sub_i in enumerate(subjects):

        pc_scores_dict = {}

        # loop over outputs
        for output in outputs:

            for time_segment in segments:
                    
                key = time_window + '_segment' + str(time_segment) + '_' + output

                # load PC scores of incorrect trials
                path_pca_sub =   os.path.join(path_root, 'results', pca_folder, time_window + '_time_resolved', sub_i)
                filename = os.path.join(path_pca_sub, 'pca_' + trial_type + '.pkl')
                with open(filename, 'rb') as file:
                    pca_dict = pickle.load(file)

                # load transforms of correct trials
                path_transforms = os.path.join(path_root, 'results', pca_aligned_folder, time_window + '_time_resolved', sub_i)
                filename = os.path.join(path_transforms, 'transforms_correct_trials.pkl')
                with open(filename, 'rb') as file:
                    transforms = pickle.load(file)

                pc_scores_dict['pc_scores_' + key] = pca_dict['pc_scores_' + key][:,:number_components_align] @ transforms[key]

        # Save the dictionary to a file using pickle
        path_pc_scores_aligned =   os.path.join(path_root, 'results', pca_aligned_folder, time_window + '_time_resolved', sub_i)
        if not os.path.isdir(path_pc_scores_aligned):
            os.makedirs(path_pc_scores_aligned, exist_ok=True)
            
        with open(os.path.join(path_pc_scores_aligned, 'pca_' + trial_type + '.pkl'), 'wb') as file:
            pickle.dump(pc_scores_dict, file)
